# Fine Tuning 

- Rodado em: https://colab.research.google.com/drive/1UXTbqkUaAp-qYKDlbPn3vqOSIH38I2wb?usp=sharing

## Instalar Roboflow

In [1]:
!pip install roboflow # install roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 92.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 134.4 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.15
    Uninstalling idna-3.15:
      Successfully uninstalled idna-3.15


In [2]:
from roboflow import Roboflow
rf = Roboflow(api_key="0ebDczdh4jeiu2qK9u0z")
project = rf.workspace("roboflow-universe-projects").project("license-plate-recognition-rxg4e")
version = project.version(13)
dataset = version.download("yolo26")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to License-Plate-Recognition-13 in yolo26:: 100%|██████████| 203744/203744 [00:30<00:00, 6672.84it/s]


## Treino

In [3]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 34.9 MB/s eta 0:00:00


In [4]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [5]:
import json
import os
from pathlib import Path

results = model.train(
    data=os.path.join(dataset.location, "data.yaml"),
    epochs=1,
    imgsz=640,
    batch=16,
    workers=2,
    device=0
)

Ultralytics 8.4.55 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/License-Plate-Recognition-13/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True,

## Copiamos os pesos para um arquivo


In [7]:
import shutil

In [8]:
os.makedirs("saved_models", exist_ok=True)
shutil.copy("runs/detect/train/weights/best.pt", "runs/detect/train/weights/license_plate_best.pt")
shutil.copy("runs/detect/train/weights/last.pt", "runs/detect/train/weights/license_plate_last.pt")
if os.path.exists("runs/detect/train/weights/license_plate_best.pt"):
    print("Pesos salvos com sucesso!")

Pesos salvos com sucesso!


## Extraindo metricas do modelo

Esta etapa roda a validacao do melhor checkpoint e salva um resumo com as principais metricas de deteccao.

In [9]:
best_weights = Path("runs/detect/train/weights/best.pt")
metrics_output = Path("saved_models/metrics_summary.json")

if not best_weights.exists():
    raise FileNotFoundError(f"Checkpoint nao encontrado: {best_weights}")

best_model = YOLO(str(best_weights))
val_metrics = best_model.val(
    data=os.path.join(dataset.location, "data.yaml"),
    split="val",
    imgsz=640,
    batch=16,
    device=0,
    plots=True,
)

box_metrics = val_metrics.box
metrics_summary = {
    "weights": str(best_weights),
    "dataset_yaml": os.path.join(dataset.location, "data.yaml"),
    "split": "val",
    "precision": float(box_metrics.p),
    "recall": float(box_metrics.r),
    "map50": float(box_metrics.map50),
    "map75": float(box_metrics.map75),
    "map50_95": float(box_metrics.map),
    "fitness": float(val_metrics.fitness),
    "speed_ms": {k: float(v) for k, v in val_metrics.speed.items()},
    "per_class_map50_95": {
        str(best_model.names[idx]): float(score)
        for idx, score in enumerate(box_metrics.maps)
    },
    "raw_results": {
        key: float(value)
        for key, value in val_metrics.results_dict.items()
    },
}

metrics_output.write_text(
    json.dumps(metrics_summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(json.dumps(metrics_summary, indent=2, ensure_ascii=False))
print(f"Metricas salvas em: {metrics_output}")

Ultralytics 8.4.55 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 122 layers, 2,375,031 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 714.4±263.3 MB/s, size: 19.8 KB)
val: Scanning /content/License-Plate-Recognition-13/valid/labels.cache... 2048 images, 3 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2048/2048 452.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 128/128 7.6it/s 16.9s
                   all       2048       2195      0.942      0.893      0.951      0.637
Speed: 0.9ms preprocess, 3.2ms inference, 0.0ms loss, 0.2ms postprocess per image
Results saved to /content/runs/detect/val
{
  "weights": "runs/detect/train/weights/best.pt",
  "dataset_yaml": "/content/License-Plate-Recognition-13/data.yaml",
  "split": "val",
  "precision": 0.9423551352515454,
  "recall": 0.8933940774487471,
  "map50": 0.951211390952955,
  "map75": 0.75

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
